# Time series

*Figures are omitted; the API is the focus here.*

In [ ]:
import numpy as np

import mefikit as mf

## Creating a time series

A time series is a set of fields sharing a common **base name**, each step stored as a field
named `"<base>_step_<n>_time_<t>"`. Steps are numbered automatically (`0, 1, 2, ...`), and the
time stamps are yours to choose: they need not be sorted nor unique.

In [ ]:
mesh = mf.build_cmesh(np.linspace(0.0, 1.0, 3), np.linspace(0.0, 1.0, 3))

In [ ]:
mesh.time_series["T"].add(20.0, time=0.0)

In [ ]:
mesh.time_series["T"].add(40.0, time=0.5)
mesh.time_series["T"].add(60.0, time=1.0)

In [ ]:
mesh.time_series

In [ ]:
mesh.fields.keys()  # the same data, seen as raw steps

## Step access

`mesh.time_series` maps base names to `TimeSeries` handles. A handle knows its base, the
dimension it lives on, and its ordered steps.

In [ ]:
s = mesh.time_series["T"]

In [ ]:
s.base, s.dim()

In [ ]:
s.steps()

In [ ]:
s.times()

In [ ]:
s.at(0).numpy()  # exact step

In [ ]:
s.at_time(0.5).numpy()  # exact time

In [ ]:
s.nearest(0.7).numpy()  # closest time

In [ ]:
s.latest().numpy()  # last step

In [ ]:
s[1]  # int -> step
s.nearest(0.9)  # float lookups go through nearest()/at_time()

`TimeSeries` also supports the usual collection protocol.

In [ ]:
list(s)  # (time, FieldRef) pairs, in step order

In [ ]:
s.items()  # (step, time, FieldRef) triples

In [ ]:
(len(s), 1 in s, 7 in s)

In [ ]:
s  # compact repr

## Growing a series

`add` appends the next step (`max step + 1`) with the given time. Values accept the same inputs
as `fields`: scalars, arrays, per-block dicts, field expressions,

In [ ]:
mesh.time_series["T"].add(np.array([1.0, 2.0, 3.0, 4.0]), time=1.5)

In [ ]:
s.times()

In [ ]:
s.at(3).numpy()

## Mutating steps

Timestamps can be rewritten in place, and steps removed again.

In [ ]:
s.set_time(0, -0.1)  # re-stamp step 0

In [ ]:
s.times()

In [ ]:
s.remove(1)
print(s.steps(), s.times())

`TimeSeries` is also subscript-settable / deletable through dunder methods.

In [ ]:
del s[3]
print(s.steps(), s.times())

## Renaming a series

Renaming moves every step to a new base name, and the handle follows.

In [ ]:
s.rename("Temperature")
print(s.base)

In [ ]:
mesh.time_series

In [ ]:
sorted(mesh.fields.keys())

## Coexisting with plain fields

Static, non-stepped fields live alongside the series and are untouched by any of the
time-series helpers.

In [ ]:
mesh.fields["steady"] = 3.0
print(sorted(mesh.fields.keys()))

## Limitations

Time series are a light naming scheme over ordinary fields: they do **not** model MED's
multi-step fields. MED import/export currently only handle the first step of a series; a
conversion with time-stepping is not provided (yet).